# Jour 5 — Finalisation : résultats finaux, prédiction 

## Étape 1 — Importer et configurer


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

CLASSES = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

if os.path.exists("/kaggle/working"):
    BASE = Path("/kaggle/working")
else:
    BASE = Path(".")

SPLIT_DIR = BASE / "data" / "split"
MODELS_DIR = BASE / "models"
RESULTS_DIR = BASE / "results"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Nouvelle session Kaggle : chercher le split dans les Inputs
if not SPLIT_DIR.exists() and os.path.exists("/kaggle/input"):
    for dossier, sous_dossiers, fichiers in os.walk("/kaggle/input"):
        if dossier.endswith("split") and "test" in sous_dossiers:
            SPLIT_DIR = Path(dossier)
            break
assert SPLIT_DIR.exists(), "Dossier split introuvable : exécutez d'abord le Jour 1."

# Dossier seg_pred (images sans étiquette)
PRED_DIR = None
if os.path.exists("/kaggle/input"):
    for dossier, sous_dossiers, fichiers in os.walk("/kaggle/input"):
        if "seg_pred" in dossier and any(f.lower().endswith((".jpg", ".png")) for f in fichiers):
            PRED_DIR = Path(dossier)
            break
elif Path("data/raw/seg_pred").exists():
    PRED_DIR = Path("data/raw/seg_pred")


def trouver_fichier(nom):
    chemin = MODELS_DIR / nom
    if chemin.exists():
        return chemin
    if os.path.exists("/kaggle/input"):
        for dossier, sous_dossiers, fichiers in os.walk("/kaggle/input"):
            if nom in fichiers:
                return Path(dossier) / nom
    raise FileNotFoundError(nom + " introuvable : exécutez le Jour 4 ou ajoutez sa sortie avec Add Input.")


sns.set_theme()
print("Dossier split   :", SPLIT_DIR)
print("Dossier seg_pred:", PRED_DIR)


test_ds = keras.utils.image_dataset_from_directory(
    str(SPLIT_DIR / "test"), label_mode="int", class_names=CLASSES,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)
test_files = test_ds.file_paths
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

nom_final = open(trouver_fichier("final_model_name.txt")).read().strip()
final_model = keras.models.load_model(str(trouver_fichier("final_model.keras")))

print("Modèle retenu :", nom_final)

## Étape 2 — Performance finale sur le test

**Problème :** donner les **chiffres définitifs** à présenter .

**Explication du code :** exactement comme au Jour 4 :
- `predict` donne les probabilités, `argmax` choisit la classe ;
- `classification_report` : precision, recall et F1 par classe ;
- la matrice de confusion est dessinée et **sauvegardée** dans `results/matrice_confusion.png` avec `plt.savefig`.

In [ ]:
y_true = np.concatenate([labels.numpy() for images, labels in test_ds])
probs = final_model.predict(test_ds)
y_pred = np.argmax(probs, axis=1)

print("Accuracy sur le test :", round(accuracy_score(y_true, y_pred), 4))
print()
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Classe prédite")
plt.ylabel("Vraie classe")
plt.title("Matrice de confusion - " + nom_final)
plt.savefig(RESULTS_DIR / "matrice_confusion.png", dpi=120, bbox_inches="tight")
plt.show()

## Étape 4 — Courbes d'apprentissage du modèle final

**Problème :** avoir l'image des courbes pour le README et la présentation.

**Explication du code :**
- On recharge l'historique du modèle final (`history_cnn_v1.csv`, etc., sauvegardé au Jour 3 ou 4).
- `try / except` : si le fichier est introuvable, le notebook continue sans planter (vous pourrez reprendre les courbes du Jour 3 ou 4).
- `plt.savefig` enregistre l'image dans `results/courbes.png`.

In [ ]:
try:
    hist = pd.read_csv(trouver_fichier("history_" + nom_final + ".csv"))

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(hist["loss"], label="train")
    plt.plot(hist["val_loss"], label="validation")
    plt.title("Loss - " + nom_final)
    plt.xlabel("époque")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(hist["accuracy"], label="train")
    plt.plot(hist["val_accuracy"], label="validation")
    plt.title("Accuracy - " + nom_final)
    plt.xlabel("époque")
    plt.legend()

    plt.savefig(RESULTS_DIR / "courbes.png", dpi=120, bbox_inches="tight")
    plt.show()
except FileNotFoundError as erreur:
    print("Historique non trouvé :", erreur)

## Étape 5 — Tableau de comparaison des configurations

**Problème :** avoir sous les yeux le tableau du Jour 4 pour le README (comparaison des 3 modèles).

**Explication du code :** on recharge `comparaison.csv` (créé à l'étape 14 du Jour 4) et on l'enregistre aussi dans `results/`.

In [ ]:
try:
    comparaison = pd.read_csv(trouver_fichier("comparaison.csv"), index_col=0)
    print(comparaison)
    comparaison.to_csv(RESULTS_DIR / "comparaison.csv")
except FileNotFoundError as erreur:
    print("Tableau non trouvé :", erreur)

## Étape 6 — La fonction de prédiction

**Problème :** donner une image **jamais vue** au système et obtenir sa **catégorie** et sa **probabilité**.

**Les 5 étapes de la fonction :**
1. **Charger** l'image et la **redimensionner** à 150 × 150 (`load_img` avec `target_size`) → même taille qu'à l'entraînement ;
2. La transformer en **tableau de nombres** (`img_to_array`) ;
3. Ajouter la dimension **batch** (`expand_dims`) : le modèle attend `(nombre d'images, 150, 150, 3)`, même pour une seule image ;
4. **Prédire** : le modèle renvoie 6 probabilités ;
5. Prendre la **plus grande** (`argmax`) : c'est la classe prédite, et sa probabilité.

In [ ]:
def predict_image(chemin_image, model):
    # 1. charger et redimensionner
    img = keras.utils.load_img(chemin_image, target_size=IMG_SIZE)
    # 2. transformer en tableau (150, 150, 3)
    tableau = keras.utils.img_to_array(img)
    # 3. ajouter la dimension batch -> (1, 150, 150, 3)
    tableau = np.expand_dims(tableau, axis=0)
    # 4. prédire : 6 probabilités
    probas = model.predict(tableau, verbose=0)[0]
    # 5. classe la plus probable
    indice = int(np.argmax(probas))
    return CLASSES[indice], float(probas[indice]), probas, img

## Étape 7 — Tester sur de nouvelles images

**Problème :** vérifier que la prédiction fonctionne sur des images que le modèle n'a jamais vues.

**Explication du code :**
- On choisit 6 images au hasard dans `seg_pred` (si ce dossier n'est pas trouvé, on prend des images de test).
- Pour chacune, on appelle `predict_image` et on affiche l'image avec le titre **classe (probabilité en %)**.
- Les images de `seg_pred` n'ont pas d'étiquette : c'est à **vous** de juger visuellement si la prédiction est correcte.
- `plt.savefig` sauvegarde la grille dans `results/exemples_predictions.png` (à mettre dans le README).

In [ ]:
if PRED_DIR is not None:
    chemins = [str(PRED_DIR / nom) for nom in os.listdir(PRED_DIR)]
else:
    chemins = test_files

np.random.seed(1)
choix = np.random.choice(chemins, size=6, replace=False)

plt.figure(figsize=(12, 8))
for k, chemin in enumerate(choix):
    classe, proba, probas, img = predict_image(chemin, final_model)
    plt.subplot(2, 3, k + 1)
    plt.imshow(img)
    plt.title(classe + " (" + str(round(proba * 100)) + " %)")
    plt.axis("off")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "exemples_predictions.png", dpi=120, bbox_inches="tight")
plt.show()

## Étape 8 — Détail d'une prédiction

**Problème :** pour la démonstration, montrer **toutes** les probabilités et signaler quand le modèle est peu sûr.

**Explication du code :**
- `np.argsort(probas)[::-1]` classe les 6 classes de la **plus probable** à la moins probable.
- Si la probabilité maximale est **inférieure à 60 %**, on affiche un avertissement : la prédiction est incertaine.
- Pour tester **votre propre image**, remplacez la variable `mon_image` par son chemin.

In [ ]:
mon_image = choix[0]        # remplacez par le chemin de votre image

classe, proba, probas, img = predict_image(mon_image, final_model)

plt.imshow(img)
plt.axis("off")
plt.title("Prédiction : " + classe + " (" + str(round(proba * 100, 1)) + " %)")
plt.show()

print("Classe prédite :", classe)
print("Probabilité    :", round(proba * 100, 1), "%")
print("Détail :")
for i in np.argsort(probas)[::-1]:
    print("  ", CLASSES[i], "->", round(probas[i] * 100, 1), "%")

if proba < 0.6:
    print("Attention : le modèle est peu sûr de sa prédiction.")